# 🔍 RAG-Pipeline mit LangGraph

**Retrieval-Augmented Generation** — Von Naïve RAG zu Agentic RAG mit Multi-Step Reasoning.

## Übersicht

Dieses Notebook demonstriert drei aufeinander aufbauende RAG-Implementierungen:
1. **Naïve RAG** — Einfache Retrieve → Generate Pipeline mit TF-IDF Vector Store
2. **LangGraph RAG** — Stateful Workflow mit Halluzinationserkennung und Rewrite-Loop
3. **Agentic RAG** — Multi-Step Reasoning mit Query-Decomposition und Evidence-Verification

> **Repository:** [github.com/mark-baumann/rag-agent-langgraph](https://github.com/mark-baumann/rag-agent-langgraph)

## 1. Umgebung & Imports

In [ ]:
import sys
import os
import math
import re
import operator
from typing import Annotated, Literal, TypedDict, Any

import numpy as np

# Projekt-Root zum Pfad hinzufügen
sys.path.insert(0, os.path.abspath("."))

# LangGraph (optional)
try:
    from langgraph.graph import END, StateGraph
    HAS_LANGGRAPH = True
    print("✅ LangGraph verfügbar")
except ImportError:
    HAS_LANGGRAPH = False
    print("⚠️  LangGraph nicht installiert — StateGraph-Demo wird simuliert")

print(f"   NumPy: {np.__version__}")

## 2. Dokumente & Embeddings

Lade PDFs/Textdokumente und erstelle Embeddings für den Vector Store.

In [ ]:
# === Dokumenten-Korpus (Gesundheitswesen / Krankenhausfinanzierung) ===
documents = [
    "Vergütungsvereinbarung 2025: Der Basisfallwert beträgt 4.200 Euro.",
    "Vergütungsvereinbarung 2026: Der Basisfallwert steigt auf 4.350 Euro.",
    "SGB V §87: Die Vergütung der Krankenhäuser richtet sich nach Fallpauschalen.",
    "Krankenhausfinanzierung: DRG-System wurde 2003 eingeführt.",
    "Qualitätsberichte: Krankenhäuser müssen jährlich Qualitätsberichte veröffentlichen.",
    "Pflegepersonaluntergrenzen: Seit 2019 gelten verbindliche Untergrenzen.",
    "Hybrid-DRG: Neue Vergütungsform für bestimmte Leistungen ab 2024.",
    "Notfallversorgung: Reform der Notfallversorgung ist in Planung.",
    "Krankenhausstrukturfonds: Förderung von Investitionen in Krankenhäuser.",
    "MDK-Reform: Der Medizinische Dienst wurde 2020 reformiert.",
    "Pflegebudget: Seit 2020 gilt das Pflegebudget zur Finanzierung der Pflege.",
    "Ambulantisierung: Verlagerung stationärer Leistungen in den ambulanten Bereich.",
]

print(f"📚 Dokumenten-Korpus: {len(documents)} Dokumente")
for i, doc in enumerate(documents):
    print(f"   [{i}] {doc}")

### 2.1 PDF-Dokumente einlesen (optional)

Falls du PDFs hast, kannst du sie hier einlesen:

In [ ]:
# === PDF-Einlesen (optional) ===
def load_pdf_texts(pdf_dir="data/pdfs/"):
    """Liest alle PDFs aus einem Verzeichnis und extrahiert Text."""
    pdf_texts = []
    if not os.path.exists(pdf_dir):
        print(f"⚠️  Verzeichnis {pdf_dir} nicht gefunden — verwende Demo-Dokumente.")
        return documents

    try:
        import pymupdf  # fitz
        for filename in sorted(os.listdir(pdf_dir)):
            if filename.endswith(".pdf"):
                filepath = os.path.join(pdf_dir, filename)
                doc = pymupdf.open(filepath)
                text = "\n".join([page.get_text() for page in doc])
                # Splitte in Abschnitte (~200 Zeichen)
                chunks = [text[i:i+500] for i in range(0, len(text), 500)]
                pdf_texts.extend(chunks)
                print(f"   📄 {filename}: {len(chunks)} Chunks")
        return pdf_texts if pdf_texts else documents
    except ImportError:
        print("⚠️  pymupdf nicht installiert — verwende Demo-Dokumente.")
        print("   Installiere mit: uv pip install pymupdf")
        return documents


# PDFs laden (oder Demo-Dokumente verwenden)
documents = load_pdf_texts()
print(f"\n📚 Verwendete Dokumente: {len(documents)}")

### 2.2 TF-IDF Embedder & Vector Store

In [ ]:
class SimpleEmbedder:
    """TF-IDF-basierter Embedder für Demo-Zwecke."""

    def __init__(self):
        self.vocab = {}
        self.idf = {}

    def fit(self, documents):
        """Baut Vokabular und IDF-Werte auf."""
        tokenized = [doc.lower().split() for doc in documents]
        all_tokens = set()
        for tokens in tokenized:
            all_tokens.update(tokens)
        self.vocab = {token: i for i, token in enumerate(sorted(all_tokens))}
        n_docs = len(documents)
        for token in self.vocab:
            df = sum(1 for tokens in tokenized if token in tokens)
            self.idf[token] = np.log((n_docs + 1) / (df + 1)) + 1
        print(f"   Vokabular: {len(self.vocab)} Tokens")

    def embed(self, text):
        """Erzeugt TF-IDF-Embedding."""
        tokens = text.lower().split()
        vec = np.zeros(len(self.vocab))
        for token in tokens:
            if token in self.vocab:
                tf = tokens.count(token) / len(tokens)
                vec[self.vocab[token]] = tf * self.idf.get(token, 1.0)
        return vec


class SimpleVectorStore:
    """Minimaler Vector Store mit Cosine-Similarity-Suche."""

    def __init__(self):
        self.documents = []
        self.embeddings = []

    def add(self, documents, embeddings):
        self.documents.extend(documents)
        self.embeddings.extend(embeddings)

    def search(self, query_embedding, k=3):
        """Cosine-Similarity-Suche."""
        if not self.embeddings:
            return []
        query_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-8)
        scores = []
        for emb in self.embeddings:
            emb_norm = emb / (np.linalg.norm(emb) + 1e-8)
            scores.append(float(np.dot(query_norm, emb_norm)))
        top_k = np.argsort(scores)[-k:][::-1]
        return [self.documents[i] for i in top_k]


# === Embedder & Vector Store initialisieren ===
embedder = SimpleEmbedder()
embedder.fit(documents)

store = SimpleVectorStore()
embeddings = [embedder.embed(doc) for doc in documents]
store.add(documents, embeddings)

print(f"✅ Vector Store: {len(store.documents)} Dokumente indiziert")
print(f"   Embedding-Dimension: {len(embeddings[0])}")

## 3. Naïve RAG — Retrieve → Generate

Die einfachste RAG-Pipeline: Query embedden → ähnliche Dokumente finden → Antwort generieren.

In [ ]:
class NaiveRAG:
    """Einfache RAG-Pipeline: Retrieve → Generate."""

    def __init__(self, store, embedder):
        self.store = store
        self.embedder = embedder

    def retrieve(self, query, k=3):
        """Retrieval-Schritt: Query → Embedding → Search."""
        query_emb = self.embedder.embed(query)
        return self.store.search(query_emb, k=k)

    def generate(self, query, docs):
        """Generation-Schritt: Kontext + Query → Antwort."""
        if not docs:
            return "Keine relevanten Dokumente gefunden."
        context = "\n".join(f"• {doc}" for doc in docs)
        return (
            f"Basierend auf {len(docs)} Dokumenten:\n\n"
            f"{context}\n\n"
            f"→ Antwort auf: \"{query}\""
        )

    def query(self, query, k=3):
        """Führt eine komplette RAG-Query aus."""
        docs = self.retrieve(query, k=k)
        answer = self.generate(query, docs)
        return {"query": query, "retrieved_docs": docs, "answer": answer}


# === Naïve RAG testen ===
rag = NaiveRAG(store, embedder)

test_queries = [
    "Wie hoch ist der Basisfallwert 2025?",
    "Was ist das DRG-System?",
    "Welche Reformen gibt es im Gesundheitswesen?",
]

for query in test_queries:
    result = rag.query(query, k=3)
    print(f"\n{'─' * 60}")
    print(f"🔍 Query: \"{query}\"")
    print(f"   📄 Gefundene Docs: {len(result['retrieved_docs'])}")
    for doc in result["retrieved_docs"]:
        print(f"      • {doc}")
    print(f"\n📝 Antwort:\n{result['answer']}")

## 4. LangGraph RAG — Stateful Workflow

Stateful RAG mit Halluzinationserkennung und automatischem Rewrite-Loop.

```
[Query] → retrieve → grade_docs → generate → check_hallucination
              ↑                                      │
              └──────── rewrite (bei Halluzination) ──┘
```

In [ ]:
class AgentState(TypedDict):
    """State für den LangGraph RAG Agenten."""
    query: str
    retrieved_docs: Annotated[list[str], operator.add]
    graded_docs: list[str]
    answer: str
    needs_rewrite: bool
    hallucination_score: float
    iteration: int


class LangGraphRAGAgent:
    """LangGraph-basierter RAG Agent mit State Machine."""

    def __init__(self, store, embedder, max_iterations=3):
        self.store = store
        self.embedder = embedder
        self.max_iterations = max_iterations

    def retrieve(self, state):
        """Retrieval-Node: Query → Embedding → Vector Search."""
        query_emb = self.embedder.embed(state["query"])
        docs = self.store.search(query_emb, k=3)
        return {"retrieved_docs": docs}

    def grade_docs(self, state):
        """Grade-Node: Bewertet Relevanz der gefundenen Dokumente."""
        query_words = set(state["query"].lower().split())
        graded = []
        for doc in state["retrieved_docs"]:
            doc_words = set(doc.lower().split())
            if len(query_words & doc_words) > 0:
                graded.append(doc)
        if not graded:
            graded = list(state["retrieved_docs"])
        return {"graded_docs": graded}

    def generate(self, state):
        """Generate-Node: Erzeugt Antwort aus relevanten Dokumenten."""
        docs = state.get("graded_docs", state["retrieved_docs"])
        if not docs:
            return {"answer": "Keine relevanten Dokumente gefunden."}
        context = "\n".join(f"• {doc}" for doc in docs)
        answer = (
            f"Basierend auf {len(docs)} Dokumenten:\n\n"
            f"{context}\n\n"
            f"→ Antwort auf: \"{state['query']}\""
        )
        return {"answer": answer}

    def check_hallucination(self, state):
        """Hallucination-Check: Prüft ob Antwort auf Dokumenten basiert."""
        answer = state.get("answer", "")
        docs = state.get("graded_docs", state.get("retrieved_docs", []))
        if not docs or not answer:
            return {"needs_rewrite": False, "hallucination_score": 0.0}
        answer_words = set(answer.lower().split())
        doc_words = set()
        for doc in docs:
            doc_words.update(doc.lower().split())
        if not answer_words:
            return {"needs_rewrite": False, "hallucination_score": 0.0}
        overlap = len(answer_words & doc_words) / len(answer_words)
        needs_rewrite = overlap < 0.3 and state["iteration"] < self.max_iterations
        return {
            "needs_rewrite": needs_rewrite,
            "hallucination_score": round(1.0 - overlap, 3),
        }

    def rewrite(self, state):
        """Rewrite-Node: Formuliert Query um für besseres Retrieval."""
        keywords = [w for w in state["query"].lower().split() if len(w) > 3]
        rewritten = f"{state['query']} (reformuliert: {' '.join(keywords)})"
        return {"query": rewritten, "iteration": state["iteration"] + 1}

    def run_manual(self, query):
        """Führt den Workflow manuell aus (ohne LangGraph)."""
        state = {
            "query": query,
            "retrieved_docs": [],
            "graded_docs": [],
            "answer": "",
            "needs_rewrite": True,
            "hallucination_score": 0.0,
            "iteration": 0,
        }

        while state["needs_rewrite"] and state["iteration"] <= self.max_iterations:
            state.update(self.retrieve(state))
            state.update(self.grade_docs(state))
            state.update(self.generate(state))
            state.update(self.check_hallucination(state))
            if state["needs_rewrite"]:
                state.update(self.rewrite(state))

        return state

    def build_graph(self):
        """Baut den LangGraph StateGraph (benötigt langgraph)."""
        if not HAS_LANGGRAPH:
            raise ImportError("LangGraph nicht installiert. Nutze run_manual() stattdessen.")
        workflow = StateGraph(AgentState)
        workflow.add_node("retrieve", self.retrieve)
        workflow.add_node("grade_docs", self.grade_docs)
        workflow.add_node("generate", self.generate)
        workflow.add_node("check_hallucination", self.check_hallucination)
        workflow.add_node("rewrite", self.rewrite)
        workflow.set_entry_point("retrieve")
        workflow.add_edge("retrieve", "grade_docs")
        workflow.add_conditional_edges(
            "grade_docs",
            lambda s: "generate" if s.get("graded_docs") else "end",
            {"generate": "generate", "end": END},
        )
        workflow.add_edge("generate", "check_hallucination")
        workflow.add_conditional_edges(
            "check_hallucination",
            lambda s: "rewrite" if s.get("needs_rewrite") else "end",
            {"rewrite": "rewrite", "end": END},
        )
        workflow.add_edge("rewrite", "retrieve")
        return workflow.compile()


# === LangGraph RAG testen ===
agent = LangGraphRAGAgent(store, embedder, max_iterations=3)

for query in test_queries:
    result = agent.run_manual(query)
    print(f"\n{'─' * 60}")
    print(f"🔍 Query: \"{query}\"")
    print(f"   📊 Iterationen: {result['iteration']}")
    print(f"   📄 Gefundene Docs: {len(result['retrieved_docs'])}")
    print(f"   ⭐ Bewertete Docs: {len(result['graded_docs'])}")
    print(f"   🎯 Halluzination-Score: {result['hallucination_score']}")
    print(f"   🔄 Rewrite nötig: {result['needs_rewrite']}")
    print(f"\n📝 Antwort:\n{result['answer']}")

## 5. Agentic RAG — Multi-Step Reasoning

Erweiterter Workflow mit Query-Decomposition, adaptiver Retrieval-Strategie und Evidence-Verification.

```
Query → Decompose → [Sub-Queries] → Adaptive Retrieve → Verify → Refine → Answer
```

In [ ]:
class QueryDecomposer:
    """Zerlegt komplexe Queries in Teilfragen."""

    def decompose(self, query):
        """Zerlegt Query in atomare Teilfragen (regelbasiert)."""
        sub_queries = [query]
        if " vs " in query.lower() or " versus " in query.lower():
            parts = re.split(r'\s+(?:vs|versus)\s+', query, flags=re.IGNORECASE)
            sub_queries.extend([p.strip() for p in parts])
        if " und " in query.lower():
            parts = query.split(" und ")
            sub_queries.extend([p.strip() for p in parts])
        return list(set(sub_queries))


class RetrievalStrategy:
    """Adaptive Retrieval-Strategie."""

    def select(self, query, previous_results):
        """Wählt Strategie basierend auf Query-Typ."""
        fact_keywords = ["wann", "wo", "wer", "wie viele", "definition"]
        if any(kw in query.lower() for kw in fact_keywords):
            return "sparse"
        if len(previous_results) < 3:
            return "dense"
        return "dense"


class EvidenceVerifier:
    """Überprüft, ob Antworten durch Quellen gestützt werden."""

    def verify(self, answer, retrieved_docs):
        """Prüft jeden Fakt in der Antwort gegen die Quellen."""
        claims = self._extract_claims(answer)
        unsupported = []
        citations = []
        for claim in claims:
            found = False
            for doc in retrieved_docs:
                content = doc.get("content", "")
                if self._claim_in_content(claim, content):
                    citations.append(doc.get("source", "unknown"))
                    found = True
                    break
            if not found:
                unsupported.append(claim)
        return {
            "verified": len(unsupported) == 0,
            "unsupported_claims": unsupported,
            "citations": list(set(citations)),
        }

    def _extract_claims(self, text):
        sentences = re.split(r'(?<=[.!?])\s+', text)
        if len(sentences) <= 1:
            sentences = re.split(r'(?<=[.!?])', text)
        return [s.strip() for s in sentences if len(s.strip()) > 5]

    def _claim_in_content(self, claim, content):
        claim_words = set(claim.lower().split())
        content_words = set(content.lower().split())
        overlap = claim_words & content_words
        return len(overlap) / max(len(claim_words), 1) > 0.3


class AgenticRAG:
    """Agentic RAG: Multi-Step Retrieval mit Reasoning und Verifikation."""

    def __init__(self, store, embedder, max_iterations=3):
        self.store = store
        self.embedder = embedder
        self.max_iterations = max_iterations
        self.strategy = RetrievalStrategy()
        self.verifier = EvidenceVerifier()
        self.decomposer = QueryDecomposer()

    def run(self, query):
        """Führt den vollständigen Agentic RAG-Workflow aus."""
        state = {
            "query": query,
            "sub_queries": [],
            "retrieved_docs": [],
            "answer": "",
            "citations": [],
            "confidence": 0.0,
            "needs_refinement": True,
            "iteration": 0,
        }

        # Step 1: Query Decomposition
        state["sub_queries"] = self.decomposer.decompose(query)

        # Step 2-4: Multi-Step Retrieve-Verify-Refine
        while state["needs_refinement"] and state["iteration"] < self.max_iterations:
            state["iteration"] += 1

            for sub_q in state["sub_queries"]:
                strategy = self.strategy.select(sub_q, state["retrieved_docs"])
                query_emb = self.embedder.embed(sub_q)
                docs = self.store.search(query_emb, k=5)
                state["retrieved_docs"].extend([
                    {"content": doc, "source": f"doc_{i}", "strategy": strategy}
                    for i, doc in enumerate(docs)
                ])

            # Context aufbauen
            context = "\n\n".join([d["content"] for d in state["retrieved_docs"]])

            # Generate Answer
            state["answer"] = (
                f"[Agentic RAG] Antwort auf: {query}\n\n"
                f"Kontext-basierte Generierung mit {len(context)} Zeichen Kontext."
            )

            # Verify
            verification = self.verifier.verify(state["answer"], state["retrieved_docs"])
            state["citations"] = verification["citations"]
            claims = self.verifier._extract_claims(state["answer"])
            state["confidence"] = 1.0 - (len(verification["unsupported_claims"]) / max(len(claims), 1))

            if verification["verified"] or state["iteration"] >= self.max_iterations:
                state["needs_refinement"] = False
            else:
                state["sub_queries"] = verification["unsupported_claims"]

        return state


# === Agentic RAG testen ===
agentic = AgenticRAG(store, embedder, max_iterations=3)

complex_queries = [
    "Vergütungsvereinbarung 2025 vs 2026 und DRG-System",
    "Pflegepersonaluntergrenzen und Pflegebudget",
]

for query in complex_queries:
    result = agentic.run(query)
    print(f"\n{'─' * 60}")
    print(f"🔍 Query: \"{query}\"")
    print(f"   🔀 Sub-Queries: {result['sub_queries']}")
    print(f"   📊 Iterationen: {result['iteration']}")
    print(f"   📄 Docs retrieved: {len(result['retrieved_docs'])}")
    print(f"   🎯 Confidence: {result['confidence']:.2%}")
    print(f"   📎 Citations: {result['citations']}")
    print(f"\n📝 Antwort:\n{result['answer']}")

## 6. Fragen stellen — Interaktive Query

Stelle eigene Fragen an die RAG-Pipeline:

In [ ]:
# === Interaktive Query-Zelle ===
# Ersetze den String mit deiner eigenen Frage!

my_query = "Was ist die Hybrid-DRG und wann wurde sie eingeführt?"

print(f"🔍 Deine Frage: \"{my_query}\"")
print()

# Naïve RAG
result_naive = rag.query(my_query, k=3)
print("📋 Naïve RAG:")
print(f"   Gefundene Docs: {len(result_naive['retrieved_docs'])}")
for doc in result_naive['retrieved_docs']:
    print(f"   • {doc}")

# LangGraph RAG
result_lg = agent.run_manual(my_query)
print(f"\n📋 LangGraph RAG:")
print(f"   Iterationen: {result_lg['iteration']}")
print(f"   Halluzination-Score: {result_lg['hallucination_score']}")
print(f"   Rewrite: {result_lg['needs_rewrite']}")

# Agentic RAG
result_agentic = agentic.run(my_query)
print(f"\n📋 Agentic RAG:")
print(f"   Sub-Queries: {result_agentic['sub_queries']}")
print(f"   Confidence: {result_agentic['confidence']:.2%}")
print(f"   Iterationen: {result_agentic['iteration']}")

## 7. Zusammenfassung

### RAG-Evolution

| Stufe | Name | Features |
|---|---|---|
| 1 | **Naïve RAG** | Retrieve → Generate, TF-IDF Embeddings |
| 2 | **LangGraph RAG** | Stateful Workflow, Halluzinationserkennung, Rewrite-Loop |
| 3 | **Agentic RAG** | Query-Decomposition, Adaptive Retrieval, Evidence-Verification |

### Workflow-Vergleich

```
Naïve RAG:     Query → Retrieve → Generate
LangGraph RAG: Query → Retrieve → Grade → Generate → Check → [Rewrite]
Agentic RAG:   Query → Decompose → [Sub-Queries] → Retrieve → Verify → [Refine]
```

### CLI-Aufruf

```bash
# Streamlit-App starten
streamlit run app.py

# Direkt per Python
python rag_agent.py        # Naïve RAG Demo
python langgraph_agent.py  # LangGraph RAG Demo
python agentic_rag.py      # Agentic RAG (als Modul)
```

> **Repository:** [github.com/mark-baumann/rag-agent-langgraph](https://github.com/mark-baumann/rag-agent-langgraph)